<a href="https://colab.research.google.com/github/pradityaputra888-lab/latihan-machine-learning/blob/main/pertemuan4/latihan4MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# 1.Memuat dataset Titanic

In [ ]:
titanic = sns.load_dataset("titanic")

print("Ukuran dataset:", titanic.shape)
display(titanic.head())

Ukuran dataset: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


#2.Menentukan target dan fitur

In [ ]:
y = titanic["survived"].astype(int)

fitur_terpilih = ["age", "fare", "sex", "embarked"]
X_semua = titanic.drop(columns=["survived", "alive"], errors="ignore").copy()
X_terpilih = X_semua[fitur_terpilih].copy()

print("Jumlah fitur terpilih:", X_terpilih.shape[1])
print("Jumlah semua fitur prediktor:", X_semua.shape[1])

Jumlah fitur terpilih: 4
Jumlah semua fitur prediktor: 13


#3.Membagi data train dan test

In [ ]:
X_train_semua, X_test_semua, y_train, y_test = train_test_split(
    X_semua,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_terpilih = X_train_semua[fitur_terpilih].copy()
X_test_terpilih = X_test_semua[fitur_terpilih].copy()

print("Jumlah data train:", len(X_train_semua))
print("Jumlah data test:", len(X_test_semua))

Jumlah data train: 712
Jumlah data test: 179


#4.Membuat preprocessing dan Pipeline

In [ ]:
def buat_pipeline(X_train):
    kolom_numerik = X_train.select_dtypes(include=["number"]).columns.tolist()
    kolom_kategorikal = X_train.select_dtypes(exclude=["number"]).columns.tolist()

    transformer = []

    if kolom_numerik:
        pipeline_numerik = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformer.append(("num", pipeline_numerik, kolom_numerik))

    if kolom_kategorikal:
        pipeline_kategorikal = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])
        transformer.append(("cat", pipeline_kategorikal, kolom_kategorikal))

    preprocess = ColumnTransformer(transformers=transformer)

    pipe = Pipeline(steps=[
        ("prep", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ])

    return pipe

#5.Melatih dan membandingkan kedua model

In [ ]:
# Model 1: menggunakan fitur terpilih
pipe_terpilih = buat_pipeline(X_train_terpilih)
pipe_terpilih.fit(X_train_terpilih, y_train)
prediksi_terpilih = pipe_terpilih.predict(X_test_terpilih)

# Model 2: menggunakan semua fitur prediktor
pipe_semua = buat_pipeline(X_train_semua)
pipe_semua.fit(X_train_semua, y_train)
prediksi_semua = pipe_semua.predict(X_test_semua)

# Menghitung metrik evaluasi
hasil = pd.DataFrame([
    {
        "Pendekatan": "Fitur terpilih",
        "Jumlah fitur awal": X_train_terpilih.shape[1],
        "Accuracy": round(accuracy_score(y_test, prediksi_terpilih), 4),
        "Accuracy (%)": round(accuracy_score(y_test, prediksi_terpilih) * 100, 2),
        "F1-Score": round(f1_score(y_test, prediksi_terpilih), 4)
    },
    {
        "Pendekatan": "Semua fitur",
        "Jumlah fitur awal": X_train_semua.shape[1],
        "Accuracy": round(accuracy_score(y_test, prediksi_semua), 4),
        "Accuracy (%)": round(accuracy_score(y_test, prediksi_semua) * 100, 2),
        "F1-Score": round(f1_score(y_test, prediksi_semua), 4)
    }
])

display(hasil)

,Pendekatan,Jumlah fitur awal,Accuracy,Accuracy (%),F1-Score
0,Fitur terpilih,4,0.7765,77.65,0.6970
1,Semua fitur,13,0.8268,82.68,0.7704
